In [0]:
from pyspark.sql import functions as F

def build_community_timestep_panel(df):
    """
    Aggregate raw client-level data to (community_code, datetime_utc) panel.

    Keyed on datetime_utc (gap-free, no DST artifacts). A derived
    datetime_local column is kept for calendar-feature engineering.

    Inputs : client_id, datetime_utc, datetime_local, community_code, active_kw
    Missing: nulls in active_kw are NOT imputed or clipped. They are counted
             (n_missing) and naturally skipped by sum/mean/std/min/max/median.

    Output columns per (community_code, datetime_utc):
      datetime_local     wall-clock Europe/Madrid time (for features)
      n_clients          distinct clients present (reporting null or not)
      n_reporting        clients with non-null active_kw
      n_missing          clients with null active_kw
      n_new_clients      clients whose first-ever observation is this timestep
      total_kw           sum of active_kw (nulls skipped)
      mean_kw, median_kw, std_kw, min_kw, max_kw   over non-null active_kw
      avg_kw_per_client  total_kw / n_clients
      avg_kw_reporting   total_kw / n_reporting (robust to missing)
    """
    # 1) First-ever UTC timestamp per client -> flag new clients
    first_seen = (
        df.groupBy("client_id")
          .agg(F.min("datetime_utc").alias("_first_dt"))
    )
    df = (
        df.join(first_seen, "client_id", "left")
          .withColumn("_is_new",
                      (F.col("datetime_utc") == F.col("_first_dt")).cast("int"))
          .withColumn("_is_missing", F.col("active_kw").isNull().cast("int"))
    )

    # 2) Aggregate on (community, UTC timestep); carry one local timestamp
    panel = (
        df.groupBy("community_code", "datetime_utc")
          .agg(
              F.first("datetime_local").alias("datetime_local"),
              F.countDistinct("client_id").alias("n_clients"),
              F.sum("_is_missing").alias("n_missing"),
              F.sum("_is_new").alias("n_new_clients"),
              F.sum("active_kw").alias("total_kw"),
              F.mean("active_kw").alias("mean_kw"),
              F.expr("percentile_approx(active_kw, 0.5)").alias("median_kw"),
              F.stddev("active_kw").alias("std_kw"),
              F.min("active_kw").alias("min_kw"),
              F.max("active_kw").alias("max_kw"),
          )
          .withColumn("n_reporting", F.col("n_clients") - F.col("n_missing"))
          .withColumn("avg_kw_per_client",
                      F.when(F.col("n_clients") > 0,
                             F.col("total_kw") / F.col("n_clients")))
          .withColumn("avg_kw_reporting",
                      F.when(F.col("n_reporting") > 0,
                             F.col("total_kw") / F.col("n_reporting")))
    )
    return panel

In [0]:
from pyspark.sql import functions as F

df = spark.table("datathon.shared.client_consumption")
panel = build_community_timestep_panel(df)

print(f"Panel rows: {panel.count():,}")
panel.orderBy("datetime_utc", "community_code").show(5, truncate=False)

Panel rows: 537,792
+--------------+-------------------+-------------------+---------+---------+-------------+------------------+------------------+---------+------------------+------+-----------+-----------+------------------+------------------+
|community_code|datetime_utc       |datetime_local     |n_clients|n_missing|n_new_clients|total_kw          |mean_kw           |median_kw|std_kw            |min_kw|max_kw     |n_reporting|avg_kw_per_client |avg_kw_reporting  |
+--------------+-------------------+-------------------+---------+---------+-------------+------------------+------------------+---------+------------------+------+-----------+-----------+------------------+------------------+
|AN            |2024-12-31 23:00:00|2025-01-01 00:00:00|802      |0        |802          |12213.693743999998|15.229044568578551|2.127928 |52.67095968276113 |0.0   |1009.000992|802        |15.229044568578551|15.229044568578551|
|AR            |2024-12-31 23:00:00|2025-01-01 00:00:00|264      |0     

In [0]:
(panel.groupBy("community_code")
      .agg(F.sum("n_missing").alias("total_null_readings"),
           F.sum("n_reporting").alias("total_reporting"),
           (100 * F.sum("n_missing") /
            (F.sum("n_missing") + F.sum("n_reporting"))).alias("null_pct"))
      .orderBy(F.desc("null_pct"))
      .show(20, truncate=False))


+--------------+-------------------+---------------+---------------------+
|community_code|total_null_readings|total_reporting|null_pct             |
+--------------+-------------------+---------------+---------------------+
|AS            |6048               |2792496        |0.21611237843678713  |
|CB            |2976               |2059032        |0.14432533724408442  |
|PV            |2208               |5791416        |0.03811086118118815  |
|AR            |1466               |10123934       |0.014478440357911786 |
|CT            |2476               |23650484       |0.010468034444737572 |
|AN            |2976               |35379504       |0.008410942364695747 |
|MD            |2277               |28614811       |0.007956784421950969 |
|EX            |192                |4899968        |0.003918239404427611 |
|GA            |182                |7055410        |0.0025795142349500934|
|CM            |149                |9984515        |0.0014922885737567134|
|VC            |80       

In [0]:
bounds = panel.groupBy("community_code")\
              .agg(F.min("datetime_utc").alias("first_utc"),
                   F.max("datetime_utc").alias("last_utc"),
                   F.count("*").alias("actual_rows"))

continuity = (bounds.withColumn("expected_rows",
                  (F.unix_timestamp("last_utc") - F.unix_timestamp("first_utc"))/900 + 1)
                    .withColumn("internal_gaps",
                                F.col("expected_rows") - F.col("actual_rows"))
                    .orderBy(F.desc("internal_gaps")))
continuity.show(20, truncate=False)


+--------------+-------------------+-------------------+-----------+-------------+-------------+
|community_code|first_utc          |last_utc           |actual_rows|expected_rows|internal_gaps|
+--------------+-------------------+-------------------+-----------+-------------+-------------+
|MD            |2024-12-31 23:00:00|2025-11-30 22:45:00|32056      |32064.0      |8.0          |
|RI            |2024-12-31 23:00:00|2025-11-30 22:45:00|32056      |32064.0      |8.0          |
|GA            |2024-12-31 23:00:00|2025-11-30 22:45:00|32056      |32064.0      |8.0          |
|AS            |2024-12-31 23:00:00|2025-11-30 22:45:00|32056      |32064.0      |8.0          |
|CN            |2025-02-14 23:00:00|2025-11-30 22:45:00|27736      |27744.0      |8.0          |
|CB            |2024-12-31 23:00:00|2025-11-30 22:45:00|32056      |32064.0      |8.0          |
|AR            |2024-12-31 23:00:00|2025-11-30 22:45:00|32056      |32064.0      |8.0          |
|CL            |2024-12-31 23:

In [0]:
grid = (panel.groupBy("community_code")
             .agg(F.min("datetime_utc").alias("a"), F.max("datetime_utc").alias("b"))
             .withColumn("datetime_utc",
                 F.explode(F.sequence("a", "b", F.expr("INTERVAL 15 MINUTES"))))
             .select("community_code", "datetime_utc"))

gaps = grid.join(panel.select("community_code", "datetime_utc").distinct()
                      .withColumn("ok", F.lit(1)),
                 ["community_code", "datetime_utc"], "left")\
           .filter(F.col("ok").isNull()).drop("ok")

print(f"Internal UTC gaps: {gaps.count()}")
gaps.orderBy("community_code", "datetime_utc").show(50, truncate=False)

Internal UTC gaps: 104
+--------------+-------------------+
|community_code|datetime_utc       |
+--------------+-------------------+
|AN            |2025-03-30 21:00:00|
|AN            |2025-03-30 21:15:00|
|AN            |2025-03-30 21:30:00|
|AN            |2025-03-30 21:45:00|
|AR            |2025-03-30 21:00:00|
|AR            |2025-03-30 21:15:00|
|AR            |2025-03-30 21:30:00|
|AR            |2025-03-30 21:45:00|
|AR            |2025-10-26 23:00:00|
|AR            |2025-10-26 23:15:00|
|AR            |2025-10-26 23:30:00|
|AR            |2025-10-26 23:45:00|
|AS            |2025-03-30 21:00:00|
|AS            |2025-03-30 21:15:00|
|AS            |2025-03-30 21:30:00|
|AS            |2025-03-30 21:45:00|
|AS            |2025-10-26 23:00:00|
|AS            |2025-10-26 23:15:00|
|AS            |2025-10-26 23:30:00|
|AS            |2025-10-26 23:45:00|
|CB            |2025-03-30 21:00:00|
|CB            |2025-03-30 21:15:00|
|CB            |2025-03-30 21:30:00|
|CB            